# Recommendation Bot — Combined Trading Strategy

Multi-indicator strategy combining five signals into daily trade recommendations.

| # | Indicator | Role | Source |
|---|-----------|------|--------|
| 1 | Permutation Entropy | Regime filter (0=chaotic, 1=orderly) | Permutation_Entropy.ipynb |
| 2 | MACD (12/26/9) | Trend direction | MACD.ipynb |
| 3 | RSI (period=14) | Momentum — early entry signal | Permutation_Entropy.ipynb |
| 4 | Fibonacci Retracement | Support / resistance levels | Fibonacci_Retracement.ipynb |
| 5 | MA Crossover (20/50) | Trend confirmation | Permutation_Entropy.ipynb |

**Recommendation logic:** PE gates all signals. When PE=1, majority vote on {MACD, RSI, Fibonacci, MA}: 3+ agree → strong_buy/sell, 2 agree → buy/sell, else → hold.

In [85]:
import os
import csv
import math
import time
import warnings
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import requests
from scipy import stats

warnings.filterwarnings("ignore")

In [86]:
# ---- Import all indicator functions from the shared module ------------------
#
#   indicators.py consolidates every indicator used across the project:
#     indicator_1_pe           <- Permutation_Entropy.ipynb  cell c07
#     indicator_2_macd         <- MACD.ipynb                 cell 2
#     indicator_3_rsi          <- Permutation_Entropy.ipynb  cell c09
#     indicator_5_fibonacci    <- Fibonacci_Retracement.ipynb cell 2
#     indicator_4_ma_crossover <- Permutation_Entropy.ipynb  cell c10
#
from indicators import (
    indicator_1_pe,
    indicator_2_macd,
    indicator_3_rsi,
    indicator_5_fibonacci,
    indicator_4_ma_crossover,
)

print("Indicators loaded from indicators.py")
print("  indicator_1_pe           (Permutation_Entropy.ipynb)")
print("  indicator_2_macd         (MACD.ipynb)")
print("  indicator_3_rsi          (Permutation_Entropy.ipynb)")
print("  indicator_5_fibonacci    (Fibonacci_Retracement.ipynb)")
print("  indicator_4_ma_crossover (Permutation_Entropy.ipynb)")

Indicators loaded from indicators.py
  indicator_1_pe           (Permutation_Entropy.ipynb)
  indicator_2_macd         (MACD.ipynb)
  indicator_3_rsi          (Permutation_Entropy.ipynb)
  indicator_5_fibonacci    (Fibonacci_Retracement.ipynb)
  indicator_4_ma_crossover (Permutation_Entropy.ipynb)


In [87]:
# ---- Configuration ----------------------------------------------------------
API_KEY    = 'ZKMMTO1ATDBLXH2K'
TICKERS    = ['NVDA', 'JPM', 'META']
INDEX      = 'SPY'
START_DATE = '2022-01-01'
END_DATE   = '2024-12-31'
RISK_FREE  = 0.05          # annual risk-free rate

TRAIN_DAYS = 252            # walk-forward: in-sample window  (~1 year)
TEST_DAYS  = 126            # walk-forward: out-of-sample window (~6 months)

print('Config loaded. Tickers:', TICKERS)

Config loaded. Tickers: ['NVDA', 'JPM', 'META']


In [88]:
def _download_prices(ticker, start_date, end_date, api_key):
    """Download daily adjusted close prices from AlphaVantage.
    Returns list of [ticker, date, adjusted_close, log_return].
    """
    url = ('https://www.alphavantage.co/query'
           '?function=TIME_SERIES_DAILY_ADJUSTED'
           '&symbol=' + ticker +
           '&outputsize=full'
           '&apikey=' + api_key)
    response = requests.get(url)
    data     = response.json()

    ts = data.get('Time Series (Daily)', {})
    if not ts:
        print('  Warning: no data for ' + ticker + '. Keys: ' + str(list(data.keys())))
        return []

    raw = []
    for date_str, vals in ts.items():
        if start_date <= date_str <= end_date:
            raw.append([ticker, date_str, float(vals['5. adjusted close'])])
    raw.sort(key=lambda x: x[1])

    result = []
    for i, rec in enumerate(raw):
        lr = 0.0 if i == 0 else math.log(rec[2] / raw[i - 1][2])
        result.append([rec[0], rec[1], rec[2], lr])
    return result


def download_all_data(tickers, index, start_date, end_date, api_key):
    """Download + cache all ticker data. One CSV per ticker.
    Skips download if CSV already exists (respects AlphaVantage rate limits).

    Returns
    -------
    dict  {ticker: [[ticker, date, price, log_return], ...]}
    """
    all_data = {}
    for ticker in tickers + [index]:
        csv_file = ticker + '_data.csv'
        if os.path.exists(csv_file):
            print('  ' + ticker + ': loading from ' + csv_file)
            records = []
            with open(csv_file, newline='') as f:
                reader = csv.reader(f)
                next(reader)
                for row in reader:
                    records.append([row[0], row[1], float(row[2]), float(row[3])])
        else:
            print('  ' + ticker + ': downloading from AlphaVantage...')
            records = _download_prices(ticker, start_date, end_date, api_key)
            if records:
                with open(csv_file, 'w', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow(['ticker', 'date', 'adjusted_close', 'log_return'])
                    writer.writerows(records)
                print('    Saved ' + str(len(records)) + ' rows to ' + csv_file)
            else:
                print('  *** ' + ticker + ': API returned no data -- will retry next run ***')
            time.sleep(15)  # AlphaVantage free tier: 5 req/min
        all_data[ticker] = records
    return all_data

In [89]:
# Download on first run; subsequent runs load from cached CSVs
data = download_all_data(TICKERS, INDEX, START_DATE, END_DATE, API_KEY)
missing = [t for t in TICKERS + [INDEX] if not data.get(t)]
for ticker, records in data.items():
    if records:
        print("  " + ticker + ": " + str(len(records)) + " records  (" +
              records[0][1] + " to " + records[-1][1] + ")")
    else:
        print("  *** " + ticker + ": NO DATA -- re-run to retry download ***")
if missing:
    print("Skipping -- missing data for: " + str(missing))
    raise SystemExit("Re-run this cell after the rate limit resets.")

  NVDA: loading from NVDA_data.csv
  JPM: loading from JPM_data.csv
  META: loading from META_data.csv
  SPY: loading from SPY_data.csv
  NVDA: 753 records  (2022-01-03 to 2024-12-31)
  JPM: 753 records  (2022-01-03 to 2024-12-31)
  META: 753 records  (2022-01-03 to 2024-12-31)
  SPY: 753 records  (2022-01-03 to 2024-12-31)


---
## Strategy

PE acts as the gate — if the market is chaotic (PE = 0), no trade is placed.  
When PE = 1 (orderly), a majority vote of four directional indicators determines direction:

- **MACD** — trend direction from exponential moving averages
- **RSI** — momentum (fires early; RSI > 50 = gains dominating)
- **Fibonacci** — support / resistance at 38.2% and 61.8% retracement levels
- **MA Crossover** — trend confirmation from simple moving averages (20/50)

Positions are held until the signal **actively reverses** direction.

In [90]:
def get_recommendation(pe_sig, macd_sig, fib_sig, ma_sig, rsi_sig=0):
    """Combine indicator signals into a 5-level recommendation (README item 7).

    Decision tree
    -------------
    PE = 0  -> hold  (chaotic regime -- all other signals ignored)
    PE = 1  -> majority vote on {MACD, RSI, Fibonacci, MA Crossover}:
        bullish >= 3  => strong_buy
        bullish == 2  => buy
        bearish >= 3  => strong_sell
        bearish == 2  => sell
        else          => hold

    RSI (momentum mode, threshold=50) fires earlier in a trend than the
    slower MA crossover, allowing entry before the full MA lag resolves.
    """
    if pe_sig == 0:
        return "hold"

    bullish = sum(1 for s in [macd_sig, fib_sig, ma_sig, rsi_sig] if s ==  1)
    bearish = sum(1 for s in [macd_sig, fib_sig, ma_sig, rsi_sig] if s == -1)

    if   bullish >= 3: return "strong_buy"
    elif bullish == 2: return "buy"
    elif bearish >= 3: return "strong_sell"
    elif bearish == 2: return "sell"
    return "hold"

# Quick sanity check
print(get_recommendation(1,  1,  0,  1,  1))   # strong_buy  (MACD+MA+RSI=3)
print(get_recommendation(1,  1,  0,  0,  1))   # buy         (MACD+RSI=2)
print(get_recommendation(1, -1,  0, -1, -1))   # strong_sell (MACD+MA+RSI=3)
print(get_recommendation(0,  1,  1,  1,  1))   # hold        (PE=0)

strong_buy
buy
strong_sell
hold


In [91]:
def execute_trade(recommendation, log_return_next_day):
    """Execute a single trade based on a recommendation (README item 8).

    Returns (log_return, direction) or (None, None) for 'hold'.
    """
    if recommendation in ('strong_buy', 'buy'):
        return log_return_next_day, 'long'
    elif recommendation in ('strong_sell', 'sell'):
        return -log_return_next_day, 'short'
    return None, None


def execute_trades(data_dict, tickers, index_ticker='SPY', long_only=True,
                   force_close_at_end=False):
    """Run the full indicator pipeline for each ticker (README item 9).

    Opens a long on bullish signals; closes on bearish signals.
    long_only=True: bearish signals exit longs but never open shorts.
    force_close_at_end=True: close any open position at the last bar --
      use this in walk-forward folds so all deployed capital is accounted for.

    Returns
    -------
    trade_log_returns        : list[float]
    trade_dates              : list[tuple]  (entry_date, exit_date, ticker, direction)
    market_returns_at_trades : list[float]  market return on each exit day
    """
    trade_log_returns        = []
    trade_dates              = []
    market_returns_at_trades = []
    spy_ret = {r[1]: r[3] for r in data_dict.get(index_ticker, [])}

    WARMUP = 60   # max warmup: Fibonacci=60, MA=50, MACD=35, PE=32

    for ticker in tickers:
        rows   = data_dict[ticker]
        prices = [r[2] for r in rows]
        dates  = [r[1] for r in rows]

        position    = 0
        entry_price = None
        entry_date  = None

        for i in range(WARMUP, len(prices)):
            pe  = indicator_1_pe(prices, i)
            mac = indicator_2_macd(prices, i)
            fib = indicator_5_fibonacci(prices, i)
            ma  = indicator_4_ma_crossover(prices, i)
            rsi = indicator_3_rsi(prices, i)
            rec = get_recommendation(pe, mac, fib, ma, rsi)

            desired = (1  if rec in ('strong_buy',  'buy')
                  else -1 if rec in ('strong_sell', 'sell')
                  else 0)

            # In long_only mode a bearish signal closes an existing long
            # but is treated as flat for entry -- never open short.
            entry_signal = desired if not long_only else max(desired, 0)

            # Close on active signal reversal
            if position != 0 and desired == -position:
                log_ret = math.log(prices[i] / entry_price)
                if position == -1:
                    log_ret = -log_ret
                trade_log_returns.append(log_ret)
                direction = 'long' if position == 1 else 'short'
                trade_dates.append((entry_date, dates[i], ticker, direction))
                market_returns_at_trades.append(spy_ret.get(dates[i], 0.0))
                position    = 0
                entry_price = None

            # Open new position (long_only: only enter on bullish signal)
            if position == 0 and entry_signal != 0:
                position    = entry_signal
                entry_price = prices[i]
                entry_date  = dates[i]

        # Force-close any open position at the last bar in the window.
        if force_close_at_end and position != 0 and entry_price is not None:
            log_ret = math.log(prices[-1] / entry_price)
            if position == -1:
                log_ret = -log_ret
            trade_log_returns.append(log_ret)
            direction = 'long' if position == 1 else 'short'
            trade_dates.append((entry_date, dates[-1], ticker, direction))
            market_returns_at_trades.append(spy_ret.get(dates[-1], 0.0))

    return trade_log_returns, trade_dates, market_returns_at_trades

In [92]:
def compute_performance(trade_log_returns, trade_dates, market_returns_at_trades,
                        rf_annual=RISK_FREE, label='Strategy',
                        benchmark_ann_return=None):
    """Compute and print all required performance metrics.

    Parameters
    ----------
    trade_log_returns        : list[float]   log-return for each closed trade
    trade_dates              : list[tuple]   (entry_date, exit_date, ticker, direction)
    market_returns_at_trades : list[float]   market log-return on each exit day
    rf_annual                : float         annual risk-free rate (default 5%)
    label                    : str           report header label

    Metrics
    -------
    a. Number of trades per month
    b. Average return + t-test for statistical significance
    c. Average return for long trades
    d. Average return for short trades
    e. Cumulative return, annualised
    f. Sharpe Ratio, annualised
    g. Sortino Ratio, annualised
    h. Jensen's Alpha
    i. VaR at the 5% level
    """
    if not trade_log_returns:
        print('No trades executed.')
        return {}

    rets    = np.array(trade_log_returns)
    mkt     = np.array(market_returns_at_trades)
    n       = len(rets)
    long_r  = [r for r, (_, _, _, d) in zip(trade_log_returns, trade_dates) if d == 'long']
    short_r = [r for r, (_, _, _, d) in zip(trade_log_returns, trade_dates) if d == 'short']

    # a. Trades per month
    months = {}
    for _, exit_d, _, _ in trade_dates:
        months[exit_d[:7]] = months.get(exit_d[:7], 0) + 1
    avg_tpm = n / max(len(months), 1)

    # b. Average return + t-test
    mean_ret      = float(np.mean(rets))
    t_stat, p_val = stats.ttest_1samp(rets, 0)

    # c/d. Long / short averages
    avg_long  = float(np.mean(long_r))  if long_r  else float('nan')
    avg_short = float(np.mean(short_r)) if short_r else float('nan')

    # e. Cumulative & annualised return
    cum_ret = float(np.sum(rets))
    d0      = datetime.strptime(trade_dates[0][0],  '%Y-%m-%d')
    d1      = datetime.strptime(trade_dates[-1][1], '%Y-%m-%d')
    years   = max((d1 - d0).days / 365.25, 1e-6)
    ann_ret = cum_ret / years

    # f. Sharpe ratio (annualised)
    rf_d   = rf_annual / 252
    excess = rets - rf_d
    sharpe = float(np.mean(excess) / np.std(excess, ddof=1) * np.sqrt(252)) \
             if np.std(excess) > 0 else 0.0

    # g. Sortino ratio (annualised)
    down    = rets[rets < 0]
    d_std   = float(np.std(down, ddof=1)) if len(down) > 1 else 1e-9
    sortino = float((mean_ret / d_std) * np.sqrt(252))

    # h. Jensen's Alpha — only market returns on trade close dates
    if len(mkt) > 1 and np.var(mkt) > 0:
        cov   = np.cov(rets, mkt)
        beta  = cov[0, 1] / np.var(mkt, ddof=1)
        alpha = (mean_ret - (rf_d + beta * (np.mean(mkt) - rf_d))) * 252
    else:
        beta  = float('nan')
        alpha = float('nan')

    # i. VaR 5%
    var5 = float(np.percentile(rets, 5))

    # ---- Print report -------------------------------------------------------
    SEP = '=' * 56
    sig = ('***' if p_val < 0.01 else
           '**'  if p_val < 0.05 else
           '*'   if p_val < 0.10 else 'n.s.')
    print(SEP)
    print('  PERFORMANCE REPORT — ' + label)
    print(SEP)
    print('Total Trades           : ' + str(n) +
          '  (Long: ' + str(len(long_r)) + ', Short: ' + str(len(short_r)) + ')')
    print('')
    print('a. Avg Trades / Month  : ' + str(round(avg_tpm, 2)))
    print('')
    print('b. Mean Log-Return     : ' + str(round(mean_ret, 6)))
    print('   t-stat              : ' + str(round(float(t_stat), 4)))
    print('   p-value             : ' + str(round(float(p_val), 4)) + '  (' + sig + ')')
    print('')
    print('c. Avg Return (Longs)  : ' + str(round(avg_long,  6)))
    print('d. Avg Return (Shorts) : ' + str(round(avg_short, 6)))
    print('')
    print('e. Cumulative Return   : ' + str(round(cum_ret * 100, 2)) + '%')
    print('   Annualised Return   : ' + str(round(ann_ret * 100, 2)) + '%')
    if benchmark_ann_return is not None:
        diff = ann_ret - benchmark_ann_return
        sign = '+' if diff >= 0 else ''
        print('   SPY Annualised      : ' + str(round(benchmark_ann_return * 100, 2)) + '%')
        print('   Alpha vs SPY        : ' + sign + str(round(diff * 100, 2)) + '%')
    print('')
    print('f. Sharpe Ratio        : ' + str(round(sharpe,  4)))
    print('')
    print('g. Sortino Ratio       : ' + str(round(sortino, 4)))
    print('')
    print('h. Beta                : ' + str(round(float(beta),  4)))
    print('   Jensens Alpha       : ' + str(round(float(alpha), 6)))
    print('')
    print('i. VaR (5%)            : ' + str(round(var5 * 100, 2)) + '%')
    print(SEP)

    return dict(n_trades=n, avg_tpm=avg_tpm, mean_return=mean_ret,
                t_stat=float(t_stat), p_value=float(p_val),
                avg_long=avg_long, avg_short=avg_short,
                cum_return=cum_ret, ann_return=ann_ret,
                sharpe=sharpe, sortino=sortino,
                beta=float(beta), jensens_alpha=float(alpha), var5=var5)

In [93]:
def walk_forward_backtest(data_dict, tickers, index_ticker='SPY',
                          train_days=TRAIN_DAYS, test_days=TEST_DAYS):
    """Walk-forward out-of-sample validation.

    Slides a training window forward by test_days at each step.
    Strategy is applied only to the out-of-sample window.
    No parameter refitting — tests stability with fixed hyperparameters.

    Parameters
    ----------
    data_dict    : dict  {ticker: [[ticker, date, price, log_return], ...]}
    tickers      : list[str]
    index_ticker : str
    train_days   : int  in-sample window (default 252 ≈ 1 year)
    test_days    : int  out-of-sample window (default 126 ≈ 6 months)
    """
    print('=' * 60)
    print('           WALK-FORWARD BACKTEST RESULTS')
    print('  Train: ' + str(train_days) + ' days  |  Test: ' + str(test_days) + ' days')
    print('-' * 60)

    n_dates = min(len(v) for v in data_dict.values())
    folds   = []
    start   = train_days
    while start + test_days <= n_dates:
        folds.append((start, start + test_days))
        start += test_days

    print('  Total folds: ' + str(len(folds)))
    print('-' * 60)
    print('  Fold   Test Start     Test End    Ann.Ret%   Sharpe   Win%')
    print('-' * 60)

    all_oos = []
    for k, (ts, te) in enumerate(folds):
        fold_data = {t: data_dict[t][ts:te] for t in tickers + [index_ticker]}
        tr, td, mr = execute_trades(fold_data, tickers, index_ticker,
                                    force_close_at_end=True)
        if not tr:
            print('   ' + str(k + 1) + '   (no trades in this fold)')
            continue

        all_oos.extend(tr)
        r   = np.array(tr)
        d0  = datetime.strptime(td[0][0],  '%Y-%m-%d')
        d1  = datetime.strptime(td[-1][1], '%Y-%m-%d')
        yrs = max((d1 - d0).days / 365.25, 1e-6)
        ann = float(np.sum(r)) / yrs
        exc = r - RISK_FREE / 252
        sh  = (float(np.mean(exc) / np.std(exc, ddof=1) * np.sqrt(252))
               if np.std(exc) > 0 else 0.0)
        win = float(np.mean(r > 0)) * 100
        s   = fold_data[tickers[0]][0][1]  if fold_data[tickers[0]] else '---'
        e   = fold_data[tickers[0]][-1][1] if fold_data[tickers[0]] else '---'
        line = ('  ' + str(k + 1).rjust(4) + '   ' + s + '   ' + e +
                '   ' + str(round(ann * 100, 2)).rjust(8) + '%' +
                '   ' + str(round(sh, 3)).rjust(6) +
                '   ' + str(round(win, 1)).rjust(5) + '%')
        print(line)

    if all_oos:
        r_all  = np.array(all_oos)
        t, p   = stats.ttest_1samp(r_all, 0)
        print('-' * 60)
        print('  OOS avg return : ' + str(round(float(np.mean(r_all)), 6)) +
              '   t=' + str(round(float(t), 3)) +
              '   p=' + str(round(float(p), 4)))
    print('=' * 60)
    return all_oos if all_oos else []

---
## Results

Run the cells below to execute the strategy and view all performance metrics.

In [94]:
# ---- Compute SPY annualised return as benchmark ----------------------------
spy_records = data[INDEX]
spy_prices  = [r[2] for r in spy_records]
spy_cum     = math.log(spy_prices[-1] / spy_prices[0])
spy_d0      = datetime.strptime(spy_records[0][1],  "%Y-%m-%d")
spy_d1      = datetime.strptime(spy_records[-1][1], "%Y-%m-%d")
spy_years   = max((spy_d1 - spy_d0).days / 365.25, 1e-6)
spy_ann     = spy_cum / spy_years
print("SPY annualised log-return: " + str(round(spy_ann * 100, 2)) + "%")

# ---- Full-sample backtest: all tickers combined ----------------------------
trade_returns, trade_info, mkt_rets = execute_trades(data, TICKERS, INDEX)
perf = compute_performance(trade_returns, trade_info, mkt_rets,
                           label="All Tickers Combined",
                           benchmark_ann_return=spy_ann)

SPY annualised log-return: 8.3%
  PERFORMANCE REPORT — All Tickers Combined
Total Trades           : 18  (Long: 18, Short: 0)

a. Avg Trades / Month  : 1.5

b. Mean Log-Return     : 0.150139
   t-stat              : 2.5623
   p-value             : 0.0202  (**)

c. Avg Return (Longs)  : 0.150139
d. Avg Return (Shorts) : nan

e. Cumulative Return   : 270.25%
   Annualised Return   : 134.66%
   SPY Annualised      : 8.3%
   Alpha vs SPY        : +126.36%

f. Sharpe Ratio        : 9.5745

g. Sortino Ratio       : 60.9359

h. Beta                : 8.0642
   Jensens Alpha       : 31.587795

i. VaR (5%)            : -4.07%


In [95]:
# ---- Per-ticker performance breakdown --------------------------------------
for ticker in TICKERS:
    tr, td, mr = execute_trades(
        {ticker: data[ticker], INDEX: data[INDEX]}, [ticker], INDEX
    )
    if tr:
        compute_performance(tr, td, mr, label=ticker,
                            benchmark_ann_return=spy_ann)
    else:
        print("[" + ticker + "]  No trades generated.")

  PERFORMANCE REPORT — NVDA
Total Trades           : 10  (Long: 10, Short: 0)

a. Avg Trades / Month  : 1.43

b. Mean Log-Return     : 0.124165
   t-stat              : 1.5923
   p-value             : 0.1458  (n.s.)

c. Avg Return (Longs)  : 0.124165
d. Avg Return (Shorts) : nan

e. Cumulative Return   : 124.16%
   Annualised Return   : 72.91%
   SPY Annualised      : 8.3%
   Alpha vs SPY        : +64.61%

f. Sharpe Ratio        : 7.9803

g. Sortino Ratio       : 384.9462

h. Beta                : 4.1681
   Jensens Alpha       : 29.101829

i. VaR (5%)            : -0.68%
  PERFORMANCE REPORT — JPM
Total Trades           : 6  (Long: 6, Short: 0)

a. Avg Trades / Month  : 1.2

b. Mean Log-Return     : 0.122324
   t-stat              : 1.7749
   p-value             : 0.1361  (n.s.)

c. Avg Return (Longs)  : 0.122324
d. Avg Return (Shorts) : nan

e. Cumulative Return   : 73.39%
   Annualised Return   : 28.28%
   SPY Annualised      : 8.3%
   Alpha vs SPY        : +19.97%

f. Sharpe Ratio  

In [96]:
# ---- Walk-forward out-of-sample validation ---------------------------------
oos_returns = walk_forward_backtest(data, TICKERS, INDEX)

           WALK-FORWARD BACKTEST RESULTS
  Train: 252 days  |  Test: 126 days
------------------------------------------------------------
  Total folds: 3
------------------------------------------------------------
  Fold   Test Start     Test End    Ann.Ret%   Sharpe   Win%
------------------------------------------------------------
     1   2023-01-04   2023-07-06      74.86%    6.719    75.0%
     2   2023-07-07   2024-01-04      74.35%   10.165    66.7%
     3   2024-01-05   2024-07-08     190.23%   18.894    75.0%
------------------------------------------------------------
  OOS avg return : 0.042762   t=2.032   p=0.0695
